[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/llm-regime-allocator/blob/main/notebooks/nb02_llm_regime_playbook.ipynb)

In [ ]:
# Setup. On Colab this fetches the repo (data, results, replayable LLM cache); locally it does nothing.
# No API keys or paid calls are needed anywhere in this notebook.
import os, subprocess, sys
if 'google.colab' in sys.modules:
    REPO = '/content/llm-regime-allocator'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/FranQuant/llm-regime-allocator.git', REPO], check=True)
    sys.path.insert(0, f'{REPO}/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lra import viz
viz.apply_style()

# nb02 — The headline: an LLM regime call against 60/40

**Question.** Can a large language model, shown only a month-end macro and market snapshot, call the next quarter's regime well enough to run a portfolio? This notebook runs the reference design on our data: Claude Sonnet 5.5 reads the snapshot with the date and tickers removed, returns regime probabilities, and the fixed playbook of nb01 turns its call into weights. The model never sets a weight. Every answer is replayed from the committed cache; no API calls.

## Setup

**The agent.** One call per month-end, 2007-12 → 2026-05 (222 calls). The system prompt casts the model as a regime classifier for a multi-asset desk, fixes the four regime definitions, forbids asset, trade or weight recommendations, and requires a single JSON object: four probabilities that sum to one, a confidence in [0, 1], one to five drivers and a short rationale. Invalid answers are re-asked (up to two retries).

**The snapshot.** 13 macro indicators (level, 3- and 12-month change, 36-month z-score) and 7 asset classes named by role, never by ticker (1-, 3-, 6- and 12-month return, 6-month volatility, 12-month drawdown), plus the stock–bond correlation. No date.

**From call to weights.** Two mappings of the probabilities $p_t$ onto the playbook $w^{(k)}$:
- **top regime**, the reference design: hold the playbook of the most likely regime, $w_t=w^{(\hat k_t)}$ with $\hat k_t=\arg\max_k p_{t,k}$;
- **blend**: $w_t=\sum_k p_{t,k}\,w^{(k)}$, which keeps the model's uncertainty and trades less.

The blend was fixed in advance; the top-regime mapping was added afterwards as a replication of the reference design and is descriptive. Both are net of 2.5 bps.

**References.** 60/40; the **uniform control**, the blend with 25% on each regime (same machinery, no information); and the **oracle**, which uses the realised regime — a look-ahead ceiling, not a strategy.

## 1. The call moves with the cycle

In [ ]:
reg = viz.read('results/baselines/regimes.csv')
son = viz.read('results/llm/claude_sonnet/anonymized_run0.csv')[viz.REGIMES]
fig, ax = plt.subplots(figsize=(11, 3.6))
viz.prob_area(ax, son, reg['realised_forward'])
ax.set_title("Sonnet's regime probabilities each month (bands) and the regime that followed (strip)")
viz.regime_legend(ax, anchor=(0.5, -0.1)); plt.show()

**Reading.** The call follows the macro story: Risk-Off through 2008, Goldilocks in the slow recoveries of 2013–15 and 2023–24, Reflation in 2010–12 and 2020–21, Stagflation in 2022. The model hedges: its top regime averages 45% probability and exceeds 60% in only 8% of months. The strip shows how often the next quarter turned out differently.

## 2. A better forecast than every baseline, but not a significant one

In [ ]:
from lra import evaluation as ev
fair = viz.read('results/baselines/regimes_fair.csv')
y = reg['realised_forward'].dropna()
cols = lambda df, p: df[[f'{p}_{k}' for k in viz.REGIMES]].set_axis(viz.REGIMES, axis=1)
fc = {'uniform 25%': ev.uniform(reg.index), 'climatology': cols(fair, 'climatology'),
      'ML logit': cols(reg, 'logit'), 'ML GBT': cols(reg, 'gbt'), 'Sonnet 5.5': son}
t = pd.DataFrame({k: ev.score(p, y) for k, p in fc.items()}).T[['hit_rate', 'brier']]
d = ev.brier_per_obs(son, y) - ev.brier_per_obs(ev.uniform(y.index), y)
lo, hi = ev.block_bootstrap_ci(d)
print(f'Sonnet minus uniform, Brier: {d.mean():+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})')
viz.fmt(t, pct=['hit_rate'])

**Reading.** **Key finding:** Sonnet is the only forecaster below uniform on Brier (0.702 against 0.750, and against 0.861–0.880 for ML on the same snapshot), with a hit rate of 37%, level with the best ML model. The interval of its edge over uniform still includes zero. Two smaller models were tried and dropped: Claude Haiku 4.5 and Llama 3.1 8B call nearly the same regime every month (Llama: Reflation in 209 of 217), so they carry no information.

## 3. The portfolio: 60/40 returns at half the drawdown

In [ ]:
dr = viz.daily()
met = pd.read_csv(viz.RESULTS / 'phase5' / 'metrics.csv')
met = met[met.cost_bps == 2.5].set_index('strategy')
S = {'Sonnet · top regime': 'playbook_top__llm_sonnet_raw', 'Sonnet · blend': 'playbook__llm_sonnet_raw',
     '60/40': 'sixty_forty', 'uniform control': 'playbook__uniform', 'oracle (ceiling)': 'playbook__oracle'}
son_c = viz.MODEL_COLORS['claude_sonnet']
style = {'Sonnet · top regime': (son_c, '-'), 'Sonnet · blend': (son_c, ':'), '60/40': (viz.INK, '-'),
         'uniform control': (viz.GREYS['uniform'], '-'), 'oracle (ceiling)': (viz.GREYS['ml'], '--')}
viz.equity_drawdown({k: (dr[v], *style[k]) for k, v in S.items()}); plt.show()
tab = met.loc[list(S.values()), ['ann_return', 'ann_vol', 'sharpe', 'max_drawdown', 'turnover_per_year']]
viz.fmt(tab.set_axis(list(S), axis=0), pct1=['ann_return', 'ann_vol'], pct=['max_drawdown'], d2=['sharpe', 'turnover_per_year'])

**Reading.** **Key finding:** run the reference way (top regime), Sonnet's call earns about what 60/40 earns (8.1% a year against 8.3%) with three-quarters of the volatility and less than half the drawdown (−14% against −32%): Sharpe 0.83 against 0.66. The blend is smoother but less decisive (0.77) and trades a third as much. With no information, the same machinery (uniform control) reaches 0.61; with perfect foresight (oracle), 0.96.

## 4. Is it significant? Not on 18 years of data

Paired Sharpe differences on the same months: annualised, from monthly excess returns, with a 6-month moving-block bootstrap 95% interval.

In [ ]:
from lra.evaluation import monthly_excess, sharpe_diff_ci
px = pd.read_csv(viz.DATA / 'etf_prices.csv', parse_dates=['date'], index_col='date')
mex = monthly_excess(dr, px['BIL'].pct_change())
pairs = [('Sonnet top regime − 60/40', 'playbook_top__llm_sonnet_raw', 'sixty_forty'),
         ('Sonnet top regime − uniform control', 'playbook_top__llm_sonnet_raw', 'playbook__uniform'),
         ('Sonnet blend − uniform control', 'playbook__llm_sonnet_raw', 'playbook__uniform'),
         ('oracle − uniform control', 'playbook__oracle', 'playbook__uniform')]
ci = pd.DataFrame([dict(zip(['diff', 'lo', 'hi'], sharpe_diff_ci(mex[a], mex[b]))) for _, a, b in pairs],
                  index=[p[0] for p in pairs])
fig, ax = plt.subplots(figsize=(9, 2.8))
viz.ci_forest(ax, ci['diff'], ci.lo, ci.hi, ci.index, [son_c, son_c, son_c, viz.GREYS['ml']],
              xlabel='Sharpe difference, 95% CI')
ax.set_title('Every LLM interval includes zero; only the oracle clears it'); plt.tight_layout(); plt.show()
viz.fmt(ci, d2=['diff'], d3=['lo', 'hi'])

**Reading.** The point estimates favour the LLM (+0.18 over 60/40, +0.28 over the control), but none of the intervals excludes zero; only the oracle's does, and barely. With about 220 months, Sharpe differences below roughly 0.3 cannot be told apart from luck.

## What we can and cannot claim

- **On raw snapshots the reference design points the same way:** an LLM regime call driving a fixed playbook has a higher Sharpe than 60/40 (0.83 against 0.66, top-regime mapping added after the fact) and less than half its drawdown. This is a historical description, not evidence of skill, for two reasons:
- **It is not statistically significant** on 18 years of monthly data, as forecast or as portfolio.
- **It is not out-of-sample.** Sonnet was trained on text that covers 2008–2026. Removing the date does not mean the model cannot recognise the month.

## Next

nb03 asks how much of this result is memory: can the model tell which month it is looking at?